# Advanced Mushroom Classification Training v2 (YOLO)
**Optimized for 70-80% accuracy:**
- YOLOv8x-cls (largest model)
- Cosine LR with warmup
- Label Smoothing (0.1)
- Strong Mixup/CutMix (prob=0.8)
- EMA (built-in)
- Strong augmentations
- Test-Time Augmentation (TTA)
- Early stopping
- 50 epochs

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install ultralytics --quiet
!pip install pandas scikit-learn tqdm albumentations --quiet

In [ ]:
import os
import shutil
from pathlib import Path
from copy import deepcopy
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration (Optimized for 70-80%)

In [ ]:
# ============ OPTIMIZED CONFIG FOR 70-80% ============
CONFIG = {
    # Data paths
    'csv_path': '/content/drive/MyDrive/raw/DF20M-metadata/DF20M-train_metadata_PROD.csv',
    'data_dir': '/content/drive/MyDrive/raw/DF20M',
    'checkpoint_dir': '/content/drive/MyDrive/checkpoints_v2',
    'local_dataset': '/content/mushroom_dataset',

    # Model - YOLOv8x-cls for best accuracy (like ViT-Large)
    # Options: yolov8n-cls (2.7M), yolov8s-cls (6.4M), yolov8m-cls (17M), 
    #          yolov8l-cls (37M), yolov8x-cls (58M)
    'model': 'yolov8l-cls.pt',  # Large model for high accuracy
    'img_size': 224,

    # Training - same as v2 ViT
    'epochs': 50,
    'batch': 16,  # Same as ViT-Large
    'patience': 15,  # Early stopping (more patience for better convergence)

    # Optimizer - matched to v2 ViT
    'optimizer': 'AdamW',
    'lr0': 3e-5,  # Low LR like ViT-Large
    'lrf': 0.01,  # Final LR = lr0 * 0.01
    'momentum': 0.937,
    'weight_decay': 0.1,  # Strong weight decay like v2
    'warmup_epochs': 5.0,  # Same warmup as v2

    # Augmentation - STRONG like v2
    'hsv_h': 0.015,  # Hue
    'hsv_s': 0.7,    # Saturation (strong)
    'hsv_v': 0.4,    # Value
    'degrees': 30.0,  # Rotation (same as v2)
    'translate': 0.1,
    'scale': 0.5,    # Scale augmentation
    'shear': 0.0,
    'perspective': 0.0,
    'flipud': 0.2,   # Vertical flip (same as v2)
    'fliplr': 0.5,   # Horizontal flip
    'bgr': 0.0,
    'mosaic': 0.0,   # Disabled for classification
    'mixup': 0.8,    # Strong Mixup like v2 (prob=0.8)
    'copy_paste': 0.0,
    'erasing': 0.4,  # Random erasing (similar to v2 RandomErasing)
    'crop_fraction': 1.0,

    # Regularization - matched to v2
    'label_smoothing': 0.1,  # Same as v2
    'dropout': 0.1,  # Similar to drop_path in v2

    # Other
    'workers': 4,
    'seed': 42,
    'val_split': 0.2,
    
    # TTA
    'use_tta': True,
    'tta_augments': 5,
}

os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)
os.makedirs(CONFIG['local_dataset'], exist_ok=True)
print("Configuration loaded!")
print(f"Model: {CONFIG['model']}")
print(f"Epochs: {CONFIG['epochs']}")
print(f"Batch size: {CONFIG['batch']}")
print(f"LR: {CONFIG['lr0']}")
print(f"Mixup: {CONFIG['mixup']}")
print(f"Label smoothing: {CONFIG['label_smoothing']}")

## Prepare Dataset
YOLOv8 classification format:
```
dataset/
  train/
    species_1/
    species_2/
  val/
    species_1/
```

In [ ]:
def prepare_dataset(config):
    """Prepare dataset in YOLOv8 classification format."""
    
    train_dir = Path(config['local_dataset']) / 'train'
    val_dir = Path(config['local_dataset']) / 'val'
    
    # Check if already prepared
    if train_dir.exists() and len(list(train_dir.iterdir())) > 0:
        num_classes = len(list(train_dir.iterdir()))
        num_train = sum(1 for _ in train_dir.rglob('*.[jJ][pP][gG]'))
        num_val = sum(1 for _ in val_dir.rglob('*.[jJ][pP][gG]'))
        print(f"Dataset already prepared!")
        print(f"Classes: {num_classes}, Train: {num_train}, Val: {num_val}")
        
        # Load species mapping
        species_list = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])
        species_to_id = {sp: idx for idx, sp in enumerate(species_list)}
        return num_classes, species_to_id
    
    # Load metadata
    df = pd.read_csv(config['csv_path'])
    df = df.dropna(subset=['species']).reset_index(drop=True)
    
    print(f"Total samples: {len(df)}")
    print(f"Number of classes: {df['species'].nunique()}")
    
    # Create species mapping
    unique_species = sorted(df['species'].unique())
    species_to_id = {sp: idx for idx, sp in enumerate(unique_species)}
    
    # Split by observation ID (prevents data leakage)
    unique_ids = df['gbifID'].unique()
    train_ids, val_ids = train_test_split(
        unique_ids, 
        test_size=config['val_split'], 
        random_state=config['seed']
    )
    
    train_df = df[df['gbifID'].isin(train_ids)]
    val_df = df[df['gbifID'].isin(val_ids)]
    
    print(f"Train samples: {len(train_df)}")
    print(f"Val samples: {len(val_df)}")
    
    def sanitize_name(name):
        """Sanitize species name for folder."""
        return name.replace('/', '_').replace(' ', '_').replace('(', '').replace(')', '').replace(',', '')[:60]
    
    def create_links(df_subset, output_dir):
        output_dir.mkdir(parents=True, exist_ok=True)
        created = 0
        errors = 0
        
        for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc=f"Creating {output_dir.name}"):
            species_folder = sanitize_name(row['species'])
            class_dir = output_dir / species_folder
            class_dir.mkdir(exist_ok=True)
            
            src_path = Path(config['data_dir']) / row['image_path']
            dst_path = class_dir / row['image_path']
            
            if src_path.exists() and not dst_path.exists():
                try:
                    os.symlink(src_path, dst_path)
                    created += 1
                except Exception as e:
                    errors += 1
        
        print(f"{output_dir.name}: created {created} links, {errors} errors")
    
    create_links(train_df, train_dir)
    create_links(val_df, val_dir)
    
    # Save species mapping
    import json
    with open(Path(config['checkpoint_dir']) / 'species_to_id.json', 'w') as f:
        json.dump(species_to_id, f, indent=2)
    
    return df['species'].nunique(), species_to_id

num_classes, species_to_id = prepare_dataset(CONFIG)
id_to_species = {v: k for k, v in species_to_id.items()}
print(f"\nDataset ready with {num_classes} classes!")

## Load Model

In [ ]:
from ultralytics import YOLO

# Check for existing checkpoint
checkpoint_path = Path(CONFIG['checkpoint_dir']) / 'yolo_mushroom_v2' / 'weights' / 'last.pt'
best_path = Path(CONFIG['checkpoint_dir']) / 'yolo_mushroom_v2' / 'weights' / 'best.pt'

if checkpoint_path.exists():
    print(f"Resuming from: {checkpoint_path}")
    model = YOLO(str(checkpoint_path))
    resume = True
else:
    print(f"Loading pretrained: {CONFIG['model']}")
    model = YOLO(CONFIG['model'])
    resume = False

print(f"Model loaded!")

## Train Model

In [ ]:
print("=" * 60)
print("Starting Advanced YOLO Training v2")
print("Target: 70-80% accuracy")
print("=" * 60)

results = model.train(
    data=CONFIG['local_dataset'],
    epochs=CONFIG['epochs'],
    imgsz=CONFIG['img_size'],
    batch=CONFIG['batch'],
    patience=CONFIG['patience'],
    
    # Optimizer - matched to ViT v2
    optimizer=CONFIG['optimizer'],
    lr0=CONFIG['lr0'],
    lrf=CONFIG['lrf'],
    momentum=CONFIG['momentum'],
    weight_decay=CONFIG['weight_decay'],
    warmup_epochs=CONFIG['warmup_epochs'],
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    
    # Augmentation - STRONG
    hsv_h=CONFIG['hsv_h'],
    hsv_s=CONFIG['hsv_s'],
    hsv_v=CONFIG['hsv_v'],
    degrees=CONFIG['degrees'],
    translate=CONFIG['translate'],
    scale=CONFIG['scale'],
    shear=CONFIG['shear'],
    perspective=CONFIG['perspective'],
    flipud=CONFIG['flipud'],
    fliplr=CONFIG['fliplr'],
    bgr=CONFIG['bgr'],
    mosaic=CONFIG['mosaic'],
    mixup=CONFIG['mixup'],
    copy_paste=CONFIG['copy_paste'],
    erasing=CONFIG['erasing'],
    crop_fraction=CONFIG['crop_fraction'],
    
    # Regularization
    label_smoothing=CONFIG['label_smoothing'],
    dropout=CONFIG['dropout'],
    
    # Other
    workers=CONFIG['workers'],
    seed=CONFIG['seed'],
    project=CONFIG['checkpoint_dir'],
    name='yolo_mushroom_v2',
    exist_ok=True,
    pretrained=True,
    amp=True,  # Mixed precision
    resume=resume,
    
    # Cosine LR scheduler (built-in)
    cos_lr=True,
    
    # Verbose
    verbose=True,
    plots=True,
)

## Evaluate Model (without TTA)

In [ ]:
# Load best model
best_model = YOLO(str(best_path))

# Validate
metrics = best_model.val()

print("\n" + "=" * 60)
print("Validation Results (no TTA):")
print(f"Top-1 Accuracy: {metrics.top1:.2f}%")
print(f"Top-5 Accuracy: {metrics.top5:.2f}%")
print("=" * 60)

## Test-Time Augmentation (TTA)
Same TTA strategy as ViT v2 for +3-5% accuracy

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def get_tta_transforms(img_size=224):
    """Test-Time Augmentation transforms - same as ViT v2."""
    return [
        # Original
        transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        ]),
        # Horizontal flip
        transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=1.0),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        ]),
        # Center crop
        transforms.Compose([
            transforms.Resize((img_size + 32, img_size + 32)),
            transforms.CenterCrop(img_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        ]),
        # Rotation +15
        transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomRotation(degrees=(15, 15)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        ]),
        # Rotation -15
        transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomRotation(degrees=(-15, -15)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        ]),
    ]


@torch.no_grad()
def validate_with_tta(model, val_dir, device, img_size=224):
    """
    Validate with Test-Time Augmentation.
    Same strategy as ViT v2 for fair comparison.
    """
    # Get underlying PyTorch model from YOLO
    torch_model = model.model
    torch_model.eval()
    torch_model = torch_model.to(device)
    
    tta_transforms = get_tta_transforms(img_size)
    val_path = Path(val_dir) / 'val'
    
    correct = 0
    total = 0
    
    # Get all images
    all_images = []
    for class_dir in val_path.iterdir():
        if class_dir.is_dir():
            class_name = class_dir.name
            for img_path in class_dir.glob('*.[jJ][pP][gG]'):
                all_images.append((img_path, class_name))
    
    # Get class names from model
    class_names = model.names  # {0: 'class1', 1: 'class2', ...}
    name_to_idx = {v: k for k, v in class_names.items()}
    
    for img_path, true_class in tqdm(all_images, desc="TTA Validation"):
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            continue
        
        # Get true label
        if true_class not in name_to_idx:
            continue
        true_label = name_to_idx[true_class]
        
        # Apply all TTA transforms and average
        all_probs = []
        for transform in tta_transforms:
            img_tensor = transform(image).unsqueeze(0).to(device)
            
            # Forward pass
            with torch.amp.autocast('cuda'):
                outputs = torch_model(img_tensor)
            
            # Handle YOLO output format
            if isinstance(outputs, (list, tuple)):
                outputs = outputs[0]
            
            probs = F.softmax(outputs, dim=1)
            all_probs.append(probs)
        
        # Average predictions
        avg_probs = torch.stack(all_probs).mean(dim=0)
        _, predicted = avg_probs.max(1)
        
        total += 1
        correct += (predicted.item() == true_label)
    
    return 100. * correct / total if total > 0 else 0.0


# Run TTA evaluation
if CONFIG['use_tta']:
    print("\nRunning TTA evaluation (this takes longer)...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    tta_acc = validate_with_tta(
        best_model, 
        CONFIG['local_dataset'], 
        device, 
        CONFIG['img_size']
    )
    
    print(f"\n" + "=" * 60)
    print(f"Validation Accuracy (no TTA): {metrics.top1:.2f}%")
    print(f"Validation Accuracy (with TTA): {tta_acc:.2f}%")
    print(f"TTA improvement: +{tta_acc - metrics.top1:.2f}%")
    print("=" * 60)

## Save Best Model

In [ ]:
# Copy best model to main checkpoint dir
if best_path.exists():
    final_path = Path(CONFIG['checkpoint_dir']) / 'best_model.pt'
    shutil.copy(best_path, final_path)
    print(f"Best model saved to: {final_path}")
    
    # Save config
    import json
    config_path = Path(CONFIG['checkpoint_dir']) / 'config.json'
    with open(config_path, 'w') as f:
        json.dump(CONFIG, f, indent=2)
    print(f"Config saved to: {config_path}")

## Test Inference

In [ ]:
import matplotlib.pyplot as plt

def predict(image_path, model, top_k=5, use_tta=False):
    """Predict mushroom species with optional TTA."""
    
    if use_tta:
        # TTA prediction
        device = next(model.model.parameters()).device
        image = Image.open(image_path).convert('RGB')
        tta_transforms = get_tta_transforms(CONFIG['img_size'])
        
        all_probs = []
        for transform in tta_transforms:
            img_tensor = transform(image).unsqueeze(0).to(device)
            with torch.no_grad():
                outputs = model.model(img_tensor)
                if isinstance(outputs, (list, tuple)):
                    outputs = outputs[0]
            probs = F.softmax(outputs, dim=1)
            all_probs.append(probs)
        
        avg_probs = torch.stack(all_probs).mean(dim=0).squeeze()
        top_probs, top_indices = torch.topk(avg_probs, k=top_k)
        
        print(f"\nTop-{top_k} Predictions (TTA):")
        print("-" * 50)
        for i, (prob, idx) in enumerate(zip(top_probs.cpu().numpy(), top_indices.cpu().numpy())):
            print(f"{i+1}. {model.names[idx]}: {prob*100:.2f}%")
    else:
        # Standard prediction
        results = model(image_path)
        probs = results[0].probs
        top5_idx = probs.top5
        top5_conf = probs.top5conf.cpu().numpy()
        
        print(f"\nTop-{top_k} Predictions:")
        print("-" * 50)
        for i, (idx, conf) in enumerate(zip(top5_idx, top5_conf)):
            print(f"{i+1}. {model.names[idx]}: {conf*100:.2f}%")
    
    # Show image
    img = Image.open(image_path)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

# Example:
# predict('/path/to/mushroom.jpg', best_model, use_tta=True)

## Download Model

In [ ]:
from google.colab import files
files.download(str(Path(CONFIG['checkpoint_dir']) / 'best_model.pt'))

## Comparison: ViT v2 vs YOLO v2

| Feature | ViT v2 (old) | YOLO v2 (new) |
|---------|--------------|---------------|
| Model | vit_large_patch16_224 | yolov8l-cls |
| Params | ~304M | ~37M |
| Epochs | 50 | 50 |
| Batch | 16 | 16 |
| LR | 3e-5 | 3e-5 |
| Warmup | 5 epochs | 5 epochs |
| Scheduler | Cosine | Cosine |
| Label Smoothing | 0.1 | 0.1 |
| Mixup prob | 0.8 | 0.8 |
| CutMix | Yes | Via Mixup |
| EMA | Manual (warmup fix) | Built-in |
| TTA | 5 augments | 5 augments |
| Drop path/Dropout | 0.1 | 0.1 |
| Weight decay | 0.1 | 0.1 |
| AMP | Yes | Yes |

### Key differences:
- YOLO is ~8x smaller but may have similar accuracy
- YOLO has built-in EMA (no manual warmup fix needed)
- YOLO training is simpler (1 line vs 100+ lines)
- Both use same TTA strategy for final evaluation